# Membership Churn Analysis

## Notebook 11: Project Summary & Analytical Documentation

---

**Author:** D. 
**Date:** April 03, 2026 
**Dataset:** churn_t_db.csv (2.1GB, 18.4M rows)

---

## Context

This notebook provides a complete reference for the membership churn analysis project. It documents the purpose, inputs, outputs, and key findings of each notebook in the analytical pipeline. No code is executed here — this notebook serves as a navigational guide for reviewers, examiners, and future analysts.

**Project:** Membership Churn Analysis (under NDA) 
**Platform:** Databricks Serverless | Delta Lake | PySpark 
**Observation Window:** January 2021 to November 2025 (58 usable snapshots) 
**Dataset:** 18,461,480 rows × 12 columns (aggregated cohort-level panel, not individual records)

## Part I — Data Foundation (Notebooks 01–03)

### Notebook 01: Environment and Data Ingestion

**Purpose:** Ingest the raw CSV into the Databricks environment, create the Bronze Delta table, and validate the schema.

| Field | Value |
|-------|-------|
| Input | `churn_t_db.csv` (2.06 GB) |
| Output | Bronze Delta table at `/Volumes/workspace/rcn_churn/raw_data/delta_raw/` |
| Rows In | 18,461,480 |
| Rows Out | 18,461,480 |
| Columns | 12 |
| Key Validation | File size confirmed at 2.06 GB. Schema: 12 columns including `CM_snapshot_date`, `q_members_t`, `q_leavers_t`, `Region`, `MemCategory`, `YoB`, `YoJ`. All 60 monthly snapshots present (Jan 2021 – Dec 2025). |

### Notebook 02: Data Cleaning and Transformation

**Purpose:** Apply all cleaning rules identified in NB01, cast data types, add quality flags, and persist the cleaned dataset as the Silver layer.

| Field | Value |
|-------|-------|
| Input | Bronze Delta table (18,461,480 rows, 12 columns) |
| Output | Silver Delta table at `/Volumes/workspace/rcn_churn/silver/churn_cleaned/` |
| Rows In | 18,461,480 |
| Rows Out | 18,461,480 (no rows removed — issues flagged, not dropped) |
| Columns | 14 (12 original + `cleaning_flag` + `geo_flag`) |
| Key Transformations | YoB and YoJ cast from double to integer. 18 YoB outliers (future years 2020–2966) flagged. 1,183 birth/join age violations flagged. 307,221 MemSectorType casing inconsistencies standardised. 2 geographic anomalies (null region, Non Members) flagged. |


### Notebook 03: Missingness and Data Quality Validation

**Purpose:** Classify all missing data by mechanism (MCAR/MAR/MNAR), validate the Silver layer against the Bronze baseline, and produce the data quality evidence required for the analysis methodology chapter.

| Field | Value |
|-------|-------|
| Input | Silver Delta table (18,461,480 rows, 14 columns) |
| Output | Missingness classifications documented (no new tables written) |
| Key Findings | `q_leavers_t`: 98.9% null — MCAR (null = active member, by design). `MemSectorType`: 7.8% null — MAR/Not Applicable (related to membership category). `YoB`: 1.49% null — MAR (related to registration system vintage). `YoJ`: 0.0002% null — MCAR (analytically negligible). No MNAR variables identified. |


## Part II — Exploratory Analysis (Notebooks 04–05)


### Notebook 04: Exploratory Data Analysis Part 1 — Temporal and Distributions

**Purpose:** Identify temporal patterns, the membership surge structural break, seasonality, and distributional characteristics across the 60-month observation window.

| Field | Value |
|-------|-------|
| Input | Silver Delta table (18,461,480 rows, 14 columns) |
| Output | Temporal patterns documented, surge period boundaries confirmed |
| Key Findings | Membership grew from ~1.44M to ~1.78M weighted volume (23.6% growth). Surge period (Oct 2022 – Jun 2023) identified with 7.6x growth multiplier (2,619 to 20,016 net members/month). April 2022 snapshot confirmed compromised and excluded from churn calculations. December 2025 excluded (no leaver data). January renewal premium identified (1.44x above non-January average). |


### Notebook 05: Exploratory Data Analysis Part 2 — Segmentation and Patterns

**Purpose:** Profile membership segments by region, category, cohort, and age band. Identify candidate patterns for formal statistical testing in NB07.

| Field | Value |
|-------|-------|
| Input | Silver Delta table (18,461,480 rows, 14 columns) |
| Output | Segmentation profiles documented, 8 hypotheses flagged for NB07 |
| Key Findings | Three distinct category profiles: Nurse member (lowest churn, largest population), Nurse Support Worker (high churn, high recruitment), Student (highest churn, academic calendar driven). North-South churn gradient identified. Cohort divergence pattern observed — newer cohorts attrite faster. Five high-uplift regions flagged. 55-64 age band escalation flagged. |


## Part III — Core Analysis (Notebook 06)

### Notebook 06: Churn Analysis and Metrics

**Purpose:** Compute weighted churn rates across all segmentation dimensions using the locked SUM/SUM methodology. Produce the Gold analytical layer. Flag eight findings for formal testing.

| Field | Value |
|-------|-------|
| Input | Silver Delta table (18,461,480 rows, 14 columns) |
| Output | Gold Delta table at `/Volumes/workspace/rcn_churn/gold/churn_rates/` (17,842,006 rows, 18 columns). Gold Delta table `region_churn_clean/` (754 rows). Gold Parquet table `churn_risk_summary/` (31 rows). |
| Rows In | 18,461,480 (Silver) |
| Rows Out | 17,842,006 (Gold — April 2022 and December 2025 excluded) |
| Key Findings | Overall weighted churn rate: 0.6614% per month. Post-exclusion rate (Under 25 removed): 0.6408%. Nurse member national rate: ~0.51%. NSW: ~1.02%. Student: ~1.80%. Post-surge national uplift: 7.2%. Nurse member post-surge uplift: 11.1% (Dunn p = 0.002). Five high-uplift regions identified (Northern Ireland +26.5%, Yorkshire +17.4%, Wales +15.3%, North West +13.1%, London +12.9%). January premium: Student 2.96x, Nurse 1.15x, NSW 1.05x. North-South ratio: 1.135x. |
| Methodological Note | The weighted formula SUM(q_leavers_t) / SUM(q_members_t) was established in this notebook as the locked methodology after discovering that unweighted .mean() produced a 57.13% churn rate — a 56.5 percentage point inflation driven by small-denominator segments. |

## Part IV — Statistical Rigour (Notebook 07)


### Notebook 07: Statistical Analysis and Hypothesis Testing

**Purpose:** Apply formal inferential tests to the eight flagged findings from NB06. Confirm statistical significance, quantify effect sizes, and produce analysis-grade conclusions with BH-FDR multiple testing correction.

| Field | Value |
|-------|-------|
| Input | Gold Delta tables (churn_rates, region_churn_clean, churn_risk_summary) |
| Output | 13 confirmed drivers documented. No new Gold tables written. |
| Tests Applied | Mann-Whitney U with rank-biserial r (regional uplift, age band uplift). Kruskal-Wallis with eta-squared (omnibus tests). Kaplan-Meier survival curves with log-rank test. Cox PH with Schoenfeld diagnostics. Levene's and Brown-Forsythe (variance equality). Spearman correlation (cohort divergence). BH-FDR correction across all 15 tests (zero significance changes). |
| Key Findings | 13 drivers confirmed: 5 regional (Northern Ireland r=0.597, Yorkshire r=0.583, Wales r=0.524, North West r=0.497, London r=0.466), 3 age band (55-64 r=0.676, 45-54 r=0.521, 65+ r=0.659), cohort divergence (eta-sq=0.231), North-South gradient (r=0.458), surge cohort trajectory (HR converging from 0.83 to 0.97), Student January premium (r=1.000). 2 null results: NSW variance (Levene p=0.916), Wales dual-category (Spearman rho=-0.217, p=0.499). |
| Revised Finding | Surge cohort delayed attrition hypothesis not confirmed. Surge cohorts retain similarly to pre-surge at equivalent cohort age (Cox HR = 1.00). The effect is modest early elevation converging toward parity, not a delayed crossover. |

## Part V — Modelling and Delivery (Notebooks 08–09)

### Notebook 08: Feature Engineering and Predictive Modelling

**Purpose:** Translate the 13 confirmed drivers into predictive models. Produce survival analysis, time-series forecasts, SHAP explanations, risk segmentation, and revenue impact estimates.

| Field | Value |
|-------|-------|
| Input | Gold Delta tables (churn_rates, region_churn_clean, churn_risk_summary) |
| Output | 12 Gold Parquet tables: `feature_engineered/` (17,648,606 rows), `hazard_ratios/` (8 rows), `forecast_outputs/` (96 rows), `shap_importance/` (8 rows), `shap_segment_level/` (64 rows), `risk_segmentation/` (188 rows), `revenue_at_risk/` (188 rows), `predicted_retention/` (288 rows), `monthly_churn_trends/` (464 rows), `model_metadata/` (9 rows), `nb06_comparison/` (20 rows), `sensitivity_analysis/` (9 rows), `validation_metrics/` (8 rows) |
| Models Trained | 13 total: Cox PH refined (C-index 0.787), Negative Binomial primary (AIC 427,666), 7 Prophet forecasting models, 1 SARIMAX (Northern Ireland), 1 GBM surrogate (R² 0.687) |
| Key Findings | NB model AIC 207,584 lower than Poisson baseline. All 8 covariates significant in both Cox and NB with directional agreement. SHAP top 3: MemSectorType (31.6%), MemCategory (28.4%), age_band (17.7%) = 77.7% cumulative. 188 segments classified into 3 risk tiers (63 High, 62 Medium, 63 Low). Revenue at risk: £526,285 (base) to £789,428 (1.5x). Nurse member accounts for 83.5% of total revenue at risk. MemCategory Simpson's reversal: univariate HR 1.22 → multivariate HR 0.66. |


### Notebook 09: Visualisation Portfolio and Dashboard Exports

**Purpose:** Produce all publication-quality analysis figures (300 DPI PNG), table CSV exports, and stage Gold layer tables as Parquet for Power BI consumption. Read-only notebook — no data written to Gold or Silver.

| Field | Value |
|-------|-------|
| Input | Silver Delta table (Chapter 4 figures), 12 Gold tables (Chapters 5-6 figures) |
| Output | 14 main body figures, 7 appendix figures, 7 table CSVs, 16 Power BI Parquet exports |
| Figures Produced | Chapter 4: membership volume, net change, cohort trajectories. Chapter 5: national churn trend, category comparison (with inset), calendar month seasonality, regional uplift, age band deterioration. Chapter 6: AIC/dispersion comparison, paired forest plot, SHAP importance, forecast panels, revenue at risk, impact vs rate scatter. Appendix: missingness pattern, SHAP segment heatmap, Prophet vs SARIMAX, age band by period, predicted retention, risk tier distribution, manual vs model comparison. |
| Key Design Decisions | IBM accessible colour palette (WCAG AA tested, colour-blind simulated). Risk tier palette (Dark Red/Cyan/Purple) selected after systematic WCAG testing to avoid category colour conflicts. Chapter 4 figures source from Silver (EDA context); Chapters 5-6 from Gold. NB07 method (mean of monthly weighted rates) used for all figures requiring uplift values to ensure consistency with statistical test results. |


## Part VI — Agentic AI Layer (Notebook 10)


### Notebook 10: AI Layer Foundation

**Purpose:** Build a prototype agentic decision support system using LangChain, Claude API, and FAISS vector store. The agent retrieves pre-validated analytical summaries from the Gold layer findings and generates natural-language responses to stakeholder queries with mandatory human review.

| Field | Value |
|-------|-------|
| Platform | Developed outside Databricks due to API integration requirements (LangChain, Claude API, FAISS). |
| Architecture | Manager Agent routes queries to Insight Agent (retrospective analysis) or Forecast Agent (prospective projection). FAISS vector store embeds 47 pre-validated analytical summaries using all-MiniLM-L6-v2. Claude API generates responses. Human-in-the-loop review checkpoint before delivery. |
| Status | Framework built. Prototype functional for retrieval-augmented generation queries against confirmed findings. |
| analysis Reference | Chapter 6, Section 6.8 and Figure 6.7 (Agentic Architecture diagram — produced manually). |

## Pipeline Summary

| Notebook | Input Rows | Output Rows | Layer | Key Output |
|----------|-----------|-------------|-------|------------|
| NB01 | 18,461,480 | 18,461,480 | Bronze | Raw Delta table |
| NB02 | 18,461,480 | 18,461,480 | Silver | Cleaned + flagged |
| NB03 | 18,461,480 | — | Silver | Missingness classification |
| NB04 | 18,461,480 | — | Silver | Temporal patterns, surge boundaries |
| NB05 | 18,461,480 | — | Silver | Segmentation profiles, 8 flags |
| NB06 | 18,461,480 | 17,842,006 | Gold | Weighted churn rates, 3 Gold tables |
| NB07 | 17,842,006 | — | Gold | 13 confirmed drivers |
| NB08 | 17,842,006 | 17,648,606 | Gold | 12 Gold tables, 13 models |
| NB09 | 17,842,006 | — | Gold | 21 figures, 7 CSVs, 16 Parquet exports |
| NB10 | Gold summaries | — | Agentic | RAG prototype |

**Locked Methodology:** `churn_rate = SUM(q_leavers_t) / SUM(q_members_t)` — applied consistently across all 10 notebooks.

**Overall Weighted Churn Rate:** 0.6614% per month (confirmed in NB06, verified in NB09).

**Gold Layer:** 17 tables at `/Volumes/workspace/rcn_churn/gold/` comprising the complete analytical output set.